# FIT5196 Assessment 2 - Task 1 Teaching Notebook

**Group 024**  
**Task:** Data cleansing for dirty, missing, and outlier food-delivery order data.

This notebook is written for tutoring. It intentionally shows the reasoning step by step instead of only producing final CSV files. The goal is to help students understand the FIT5196 wrangling workflow:

`Discovery -> EDA/Profile -> Rule/Structure Design -> Cleaning/Imputation/Outlier Handling -> Validation -> Export QA`

> Teaching note: old 2025 Ass2 can be used as a reporting reference, but its warehouse/Haversine/season/sentiment business rules must not be copied into this 2026 food-delivery task.

## 1. Setup and Imports

We keep reusable implementation details in `Group024_ass2_task1.py`. The notebook imports that script so students can focus on *why* each step is needed while still seeing the exact functions used.

This mirrors professional wrangling practice: keep complex logic testable in functions, and use the notebook as a readable report.

In [1]:
from pathlib import Path
import ast
import importlib

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

import Group024_ass2_task1 as task1
importlib.reload(task1)

ROOT = Path.cwd()
print(f'Working directory: {ROOT}')
print('Helper script loaded:', Path('Group024_ass2_task1.py').exists())

Working directory: /Users/songhaifan/Documents/GitHub/teaching-materials/courses/lincoin/materials/fit5196/ass2_2026
Helper script loaded: True


## 2. Discovery: Load Every Input File

Before fixing anything, we first inspect what data sources exist. This reflects Weeks 1-5: data discovery, collection, and profiling come before cleaning.

In [2]:
dirty_raw = pd.read_csv('Group024_dirty_data.csv')
missing_raw = pd.read_csv('Group024_missing_data.csv')
outlier_raw = pd.read_csv('Group024_outlier_data.csv')
branches = pd.read_csv('branches.csv')
nodes = pd.read_csv('nodes.csv')
edges = pd.read_csv('edges.csv')

summary = pd.DataFrame({
    'dataset': ['dirty', 'missing', 'outlier', 'branches', 'nodes', 'edges'],
    'rows': [len(dirty_raw), len(missing_raw), len(outlier_raw), len(branches), len(nodes), len(edges)],
    'columns': [dirty_raw.shape[1], missing_raw.shape[1], outlier_raw.shape[1], branches.shape[1], nodes.shape[1], edges.shape[1]],
    'missing_cells': [dirty_raw.isna().sum().sum(), missing_raw.isna().sum().sum(), outlier_raw.isna().sum().sum(), branches.isna().sum().sum(), nodes.isna().sum().sum(), edges.isna().sum().sum()],
})
display(summary)

print('Order-data columns:')
display(pd.DataFrame({'column': dirty_raw.columns}))

print('Dirty data sample:')
display(dirty_raw.head(3))

,dataset,rows,columns,missing_cells
0,dirty,500,12,0
1,missing,500,12,200
2,outlier,500,12,0
3,branches,3,4,0
4,nodes,17117,3,0
5,edges,42224,6,0


Order-data columns:


,column
0,order_id
1,date
2,time
3,order_type
4,branch_code
5,order_items
6,order_price
7,customer_lat
8,customer_lon
9,customerHasloyalty?


Dirty data sample:


,order_id,date,time,order_type,branch_code,order_items,order_price,customer_lat,customer_lon,customerHasloyalty?,distance_to_customer_KM,delivery_fee
0,ORDX00699,03-08-2018,15:05:54,Lunch,BK,"[('Chicken', 2), ('Steak', 1)]",109.0000,-37.8123,144.9737,0,6.7760,12.6337
1,ORDX06260,2018-03-14,10:21:58,Breakfast,BK,"[('Cereal', 6), ('Pancake', 8), ('Coffee', 7), ('Eggs', 8)]",548.5000,-37.8175,144.9778,0,6.8330,11.8394
2,ORDI04941,2018-04-22,11:43:05,Breakfast,NS,"[('Cereal', 5), ('Pancake', 6), ('Coffee', 8), ('Eggs', 7)]",464.5000,-37.8191,144.9535,0,9.2260,16.3654


## 3. Profile the Main Quality Problems

The assignment separates three files by task:

- `dirty_data`: wrong values, at most one anomaly per row.
- `missing_data`: coverage anomalies only.
- `outlier_data`: outliers only, with respect to `delivery_fee`.

So we should not use one generic cleaning method for all files. We profile each file according to its role.

In [3]:
missing_profile = pd.DataFrame({
    'dirty_missing': dirty_raw.isna().sum(),
    'missing_missing': missing_raw.isna().sum(),
    'outlier_missing': outlier_raw.isna().sum(),
})
display(missing_profile[missing_profile.sum(axis=1) > 0])

category_profile = pd.DataFrame({
    'dirty_branch_values': pd.Series(dirty_raw['branch_code'].value_counts(dropna=False).to_dict()),
    'missing_branch_values': pd.Series(missing_raw['branch_code'].value_counts(dropna=False).to_dict()),
    'outlier_branch_values': pd.Series(outlier_raw['branch_code'].value_counts(dropna=False).to_dict()),
})
display(category_profile.fillna(''))

print('Order type counts in dirty file:')
display(dirty_raw['order_type'].value_counts().rename_axis('order_type').reset_index(name='count'))

,dirty_missing,missing_missing,outlier_missing
branch_code,0,100,0
distance_to_customer_KM,0,50,0
delivery_fee,0,50,0


,dirty_branch_values,missing_branch_values,outlier_branch_values
BK,150.0000,136.0000,184.0000
NS,134.0000,109.0000,158.0000
TP,188.0000,155.0000,158.0000
bk,8.0000,,
ns,10.0000,,
tp,10.0000,,
NaN,,100.0000,


Order type counts in dirty file:


,order_type,count
0,Breakfast,179
1,Lunch,173
2,Dinner,148


## 4. Encode Business Rules as Data Structures

Week 6 matters here: dictionaries and graphs make the solution clearer and less error-prone.

We encode:

- meal windows from `time`;
- branch code from `order_id` prefix;
- valid menu items and unit prices;
- branch graph nodes for road-network distance.

In [4]:
print('Branch prefix rule:')
display(pd.DataFrame(list(task1.BRANCH_PREFIX.items()), columns=['order_id_prefix', 'branch_code']))

print('Menu and unit prices:')
menu_rows = []
for meal, price_map in task1.MEAL_PRICES.items():
    for item, price in price_map.items():
        menu_rows.append({'order_type': meal, 'item': item, 'unit_price': price})
display(pd.DataFrame(menu_rows))

print('Branch graph nodes:')
display(pd.DataFrame(list(task1.BRANCH_NODES.items()), columns=['branch_code', 'graph_node']))

Branch prefix rule:


,order_id_prefix,branch_code
0,ORDA,BK
1,ORDK,BK
2,ORDX,BK
3,ORDC,NS
4,ORDI,NS
5,ORDZ,NS
6,ORDB,TP
7,ORDJ,TP
8,ORDY,TP


Menu and unit prices:


,order_type,item,unit_price
0,Breakfast,Cereal,21.0000
1,Breakfast,Coffee,7.5000
2,Breakfast,Eggs,22.0000
3,Breakfast,Pancake,24.2500
4,Lunch,Burger,31.0000
5,Lunch,Chicken,32.0000
6,Lunch,Fries,12.0000
7,Lunch,Salad,17.2000
8,Lunch,Steak,45.0000
9,Dinner,Fish&Chips,35.0000


Branch graph nodes:


,branch_code,graph_node
0,NS,2455254505
1,TP,1390575046
2,BK,1889485053


## 5. Quick Rule Checks on Dirty Data

Before fixing, we count the most visible symptoms. These counts are not final marks by themselves, because one wrong value can cause several downstream checks to fail. The important teaching point is to identify likely rule families.

In [5]:
def safe_normalized_date(x):
    return task1.normalize_date(x)

def safe_meal_from_time(x):
    try:
        return task1.meal_from_time(x)
    except Exception:
        return None

def safe_branch_from_order(x):
    try:
        return task1.branch_from_order_id(x)
    except Exception:
        return None

check = dirty_raw.copy()
check['expected_date'] = check['date'].map(safe_normalized_date)
check['expected_order_type'] = check['time'].map(safe_meal_from_time)
check['expected_branch_code'] = check['order_id'].map(safe_branch_from_order)
check['date_needs_normalising'] = check['date'] != check['expected_date']
check['order_type_mismatch'] = check['order_type'] != check['expected_order_type']
check['branch_code_mismatch'] = check['branch_code'].str.upper() != check['expected_branch_code']

symptoms = pd.Series({
    'date format/value suspicious': int(check['date_needs_normalising'].sum()),
    'order_type disagrees with time': int(check['order_type_mismatch'].sum()),
    'branch_code disagrees with order_id prefix': int(check['branch_code_mismatch'].sum()),
})
display(symptoms.to_frame('count'))

print('Examples of rows with date/order_type/branch symptoms:')
display(check.loc[check[['date_needs_normalising','order_type_mismatch','branch_code_mismatch']].any(axis=1),
                  ['order_id','date','expected_date','time','order_type','expected_order_type','branch_code','expected_branch_code']].head(10))

,count
date format/value suspicious,37
order_type disagrees with time,37
branch_code disagrees with order_id prefix,28


Examples of rows with date/order_type/branch symptoms:


,order_id,date,expected_date,time,order_type,expected_order_type,branch_code,expected_branch_code
0,ORDX00699,03-08-2018,2018-08-03,15:05:54,Lunch,Lunch,BK,BK
4,ORDK03173,2018-11-04,2018-11-04,18:08:27,Dinner,Dinner,tp,BK
10,ORDC01147,2018-26-08,2018-08-26,15:46:28,Lunch,Lunch,NS,NS
16,ORDK04564,2018-19-09,2018-09-19,16:37:10,Dinner,Dinner,BK,BK
18,ORDY05205,04-01-2018,2018-01-04,08:00:00,Breakfast,Breakfast,TP,TP
21,ORDI09297,2018-11-22,2018-11-22,18:49:00,Dinner,Dinner,BK,NS
22,ORDC07228,2018-11-08,2018-11-08,18:18:35,Dinner,Dinner,TP,NS
23,ORDX10183,2018-11-12,2018-11-12,18:28:43,Lunch,Dinner,BK,BK
28,ORDY07273,2018-06-22,2018-06-22,13:34:38,Breakfast,Lunch,TP,TP
31,ORDK06897,2018-03-21,2018-03-21,10:32:06,Breakfast,Breakfast,ns,BK


## 6. Build the Road Network and Check Distances

The 2026 task uses road-network distance, not Haversine distance. `edges.csv` is converted into a weighted graph and Dijkstra shortest paths are precomputed from each branch node.

This step connects Week 6 data structuring with Week 10 data integration/enrichment.

In [6]:
network = task1.RoadNetwork(task1.NODES_INPUT, task1.EDGES_INPUT)

print(f'Nodes loaded: {len(network.nodes):,}')
print(f'Edges loaded: {len(network.edges):,}')
print('Branch shortest-path tables:', {branch: len(paths) for branch, paths in network.shortest_paths.items()})

example = dirty_raw.iloc[0]
node, lat, lon = network.find_customer_node(example['customer_lat'], example['customer_lon'])
branch = task1.branch_from_order_id(example['order_id'])
distance = network.distance_km(branch, lat, lon)
print('Example row:', example['order_id'])
print('Detected branch:', branch)
print('Customer graph node:', node)
print('Graph distance km:', distance)
print('Distance in raw row:', example['distance_to_customer_KM'])

Nodes loaded: 17,117
Edges loaded: 42,224
Branch shortest-path tables: {'NS': 17117, 'TP': 17117, 'BK': 17117}
Example row: ORDX00699
Detected branch: BK
Customer graph node: 588198771
Graph distance km: 6.776
Distance in raw row: 6.776


## 7. Clean Dirty Data Step by Step

The helper function applies one-anomaly-per-row logic and protects fields that the specification says should not be changed.

Protected in dirty data:

- `order_id`
- `time`
- numeric quantities inside `order_items`
- `delivery_fee`

In [7]:
dirty_solution, dirty_repairs = task1.clean_dirty_data(dirty_raw, network)

print('Repair count by rule family:')
display(pd.Series(dirty_repairs).to_frame('rows'))

print('Before/after examples for changed columns:')
for col in ['date','order_type','branch_code','order_items','order_price','customer_lat','customer_lon','distance_to_customer_KM']:
    changed = dirty_raw[col].astype(str) != dirty_solution[col].astype(str)
    if changed.any():
        example_rows = pd.concat([
            dirty_raw.loc[changed, ['order_id', col]].rename(columns={col: f'before_{col}'}),
            dirty_solution.loc[changed, [col]].rename(columns={col: f'after_{col}'})
        ], axis=1).head(5)
        print()
        print(f'Column: {col} | changed rows: {int(changed.sum())}')
        display(example_rows)

Repair count by rule family:


,rows
date,37
order_type,37
branch_code,37
order_items,37
order_price,37
customer_coordinates,41
distance_to_customer_KM,37
unchanged,237


Before/after examples for changed columns:

Column: date | changed rows: 37


,order_id,before_date,after_date
0,ORDX00699,03-08-2018,2018-08-03
10,ORDC01147,2018-26-08,2018-08-26
16,ORDK04564,2018-19-09,2018-09-19
18,ORDY05205,04-01-2018,2018-01-04
37,ORDJ05383,2018-18-08,2018-08-18



Column: order_type | changed rows: 37


...

,order_id,before_customer_lat,after_customer_lat
32,ORDZ03318,37.8062,-37.8062
38,ORDZ06323,37.8142,-37.8142
57,ORDK02131,37.8056,-37.8056
62,ORDX01429,37.8142,-37.8142
66,ORDA02101,37.8222,-37.8222



Column: customer_lon | changed rows: 4


,order_id,before_customer_lon,after_customer_lon
200,ORDC05483,-37.8114,145.0015
210,ORDK00150,-37.8162,145.0074
254,ORDY06420,-37.8246,144.9885
374,ORDX05398,-37.8118,144.9564



Column: distance_to_customer_KM | changed rows: 37


,order_id,before_distance_to_customer_KM,after_distance_to_customer_KM
8,ORDY07813,7.6510,12.8470
12,ORDJ08517,7.9620,9.7640
19,ORDZ08993,7.0660,7.8570
25,ORDY03212,7.9750,9.3690
58,ORDJ10405,11.3230,10.3270


## 8. Validate Dirty-Data Protection and Rules

Validation is not optional. Week 11 distinguishes structural, content, and logical validation. Here we check that protected fields stayed unchanged and that business rules now hold.

In [8]:
task1.validate_dirty_protection(dirty_raw, dirty_solution)
dirty_rule_report = task1.validate_structural_rules(dirty_solution, network, 'dirty_solution_preview')
print('Dirty protection check: passed')
display(pd.Series(dirty_rule_report).to_frame('value'))

Dirty protection check: passed


,value
rows,500
missing_cells,0
valid_branch_codes,"[BK, NS, TP]"
valid_order_types,"[Breakfast, Dinner, Lunch]"


## 9. Impute Missing Data

The missing file has coverage anomalies only. We impute in dependency order:

1. `branch_code` from the protected `order_id` prefix.
2. `distance_to_customer_KM` from the graph once branch is known.
3. `delivery_fee` from a branch-specific linear model.

This is better than blindly using mean/mode because it uses domain rules and relationships in the data.

In [9]:
print('Missing values before imputation:')
display(missing_raw.isna().sum()[missing_raw.isna().sum() > 0].to_frame('missing_count'))

missing_solution, missing_repairs, missing_reports = task1.impute_missing_data(missing_raw, network)

print('Imputation count by column:')
display(pd.Series(missing_repairs).to_frame('rows'))

print('Missing values after imputation:')
remaining_missing = missing_solution.isna().sum()[missing_solution.isna().sum() > 0]
if len(remaining_missing):
    display(remaining_missing.to_frame('missing_count'))
else:
    print('No missing values remain.')

print('Examples of imputed rows:')
for col in ['branch_code','distance_to_customer_KM','delivery_fee']:
    mask = missing_raw[col].isna()
    if mask.any():
        sample = pd.concat([
            missing_raw.loc[mask, ['order_id', 'time', col]].rename(columns={col: f'before_{col}'}),
            missing_solution.loc[mask, [col]].rename(columns={col: f'after_{col}'})
        ], axis=1).head(5)
        print()
        print(f'Column: {col}')
        display(sample)

Missing values before imputation:


,missing_count
branch_code,100
distance_to_customer_KM,50
delivery_fee,50


Imputation count by column:


,rows
branch_code,100
distance_to_customer_KM,50
delivery_fee,50


Missing values after imputation:
No missing values remain.
Examples of imputed rows:

Column: branch_code


,order_id,time,before_branch_code,after_branch_code
0,ORDX00188,14:25:21,NaN,BK
1,ORDZ07051,08:10:08,NaN,NS
3,ORDC06592,12:33:48,NaN,NS
9,ORDJ05186,14:05:04,NaN,TP
14,ORDA07746,18:49:00,NaN,BK



Column: distance_to_customer_KM


,order_id,time,before_distance_to_customer_KM,after_distance_to_customer_KM
1,ORDZ07051,08:10:08,NaN,10.0390
9,ORDJ05186,14:05:04,NaN,10.6120
14,ORDA07746,18:49:00,NaN,9.2140
27,ORDY04867,08:50:42,NaN,8.6650
41,ORDB09188,11:43:05,NaN,8.5940



Column: delivery_fee


,order_id,time,before_delivery_fee,after_delivery_fee
7,ORDC10700,13:54:55,NaN,15.1941
33,ORDJ08835,13:24:30,NaN,12.0175
38,ORDJ07186,17:58:18,NaN,15.2423
42,ORDC03961,08:20:16,NaN,6.5603
61,ORDJ09090,14:35:29,NaN,13.1752


## 10. Delivery-Fee Model Diagnostics

The assignment states that delivery fee is branch-specific and depends linearly on:

- weekend/weekday;
- time of day;
- graph distance to customer;
- loyalty discount applied after the base fee.

We therefore fit one model per branch. The model is not the final learning objective; it is a tool for validation and imputation.

In [10]:
fee_report_df = pd.DataFrame([
    {
        'branch_code': r.branch_code,
        'training_rows': r.rows,
        'r2': r.r2,
        'residual_mean': r.residual_mean,
        'residual_std': r.residual_std,
        **{f'coef_{k}': v for k, v in r.coefficients.items()},
    }
    for r in missing_reports
])
display(fee_report_df.sort_values('branch_code'))

,branch_code,training_rows,r2,residual_mean,residual_std,coef_weekend,coef_time_code,coef_distance_to_customer_KM
0,BK,153,0.9946,-0.0000,0.3105,2.5258,0.9586,1.0571
1,NS,128,0.9670,0.0000,0.3360,1.9294,0.5377,1.0163
2,TP,169,0.9520,-0.0000,0.3162,1.4818,0.7360,0.8506


## 11. Detect and Remove Delivery-Fee Outliers

The outlier file contains no other anomalies, so we only remove rows based on `delivery_fee`.

The method fits branch-specific delivery-fee models, computes residuals after accounting for loyalty, and removes rows whose residuals are extreme within their branch. This is a model-based version of the Week 8 outlier workflow: detect, explain, then remove whole rows.

In [11]:
outlier_solution, outlier_removed, outlier_reports = task1.remove_outliers(outlier_raw)

print('Rows before outlier removal:', len(outlier_raw))
print('Rows after outlier removal:', len(outlier_solution))
print('Rows removed:', len(outlier_raw) - len(outlier_solution))

display(pd.Series(outlier_removed).rename_axis('branch_code').reset_index(name='rows_removed'))

outlier_report_df = pd.DataFrame([
    {
        'branch_code': r.branch_code,
        'retained_training_rows': r.rows,
        'r2': r.r2,
        'residual_mean': r.residual_mean,
        'residual_std': r.residual_std,
        **{f'coef_{k}': v for k, v in r.coefficients.items()},
    }
    for r in outlier_reports
])
display(outlier_report_df.sort_values('branch_code'))

Rows before outlier removal: 500
Rows after outlier removal: 452
Rows removed: 48


,branch_code,rows_removed
0,BK,10
1,NS,18
2,TP,20


,branch_code,retained_training_rows,r2,residual_mean,residual_std,coef_weekend,coef_time_code,coef_distance_to_customer_KM
0,BK,174,0.9839,-0.0000,0.2834,2.5676,0.9709,1.0614
1,NS,140,0.9694,0.0000,0.2779,1.9762,0.4996,0.9694
2,TP,138,0.9669,0.0000,0.2786,1.4267,0.7002,0.8605


## 12. Export Solution CSVs

Now we write the three output files. The most important submission rule is to keep exact column names. The automarker may reject files with renamed, missing, extra, or reordered columns.

In [12]:
task1.write_solution(dirty_solution, task1.DIRTY_OUTPUT)
task1.write_solution(missing_solution, task1.MISSING_OUTPUT)
task1.write_solution(outlier_solution, task1.OUTLIER_OUTPUT)

for path in [task1.DIRTY_OUTPUT, task1.MISSING_OUTPUT, task1.OUTLIER_OUTPUT]:
    print(path.name, 'written:', path.exists(), 'size bytes:', path.stat().st_size)

Group024_dirty_data_solution.csv written: True size bytes: 70411
Group024_missing_data_solution.csv written: True size bytes: 70381
Group024_outlier_data_solution.csv written: True size bytes: 63573


## 13. Final Read-Back QA

A good wrangling report finishes by reading its own outputs back in and validating them. This guards against accidental export mistakes.

In [13]:
validation = task1.validate_outputs(dirty_raw, missing_raw, outlier_raw, network)

print('Final validation summary:')
for name, report in validation.items():
    print()
    print(f'{name}')
    display(pd.Series(report).to_frame('value'))

print('CSV headers match inputs:')
for label, original, output_path in [
    ('dirty', dirty_raw, task1.DIRTY_OUTPUT),
    ('missing', missing_raw, task1.MISSING_OUTPUT),
    ('outlier', outlier_raw, task1.OUTLIER_OUTPUT),
]:
    exported = pd.read_csv(output_path)
    print(label, list(exported.columns) == list(original.columns), exported.shape)

Final validation summary:

dirty


,value
rows,500
missing_cells,0
valid_branch_codes,"[BK, NS, TP]"
valid_order_types,"[Breakfast, Dinner, Lunch]"



missing


,value
rows,500
missing_cells,0
valid_branch_codes,"[BK, NS, TP]"
valid_order_types,"[Breakfast, Dinner, Lunch]"



outlier


,value
rows,452
missing_cells,0
valid_branch_codes,"[BK, NS, TP]"
valid_order_types,"[Breakfast, Dinner, Lunch]"
rows_removed,48


CSV headers match inputs:


dirty True (500, 12)
missing True (500, 12)


outlier True (452, 12)


## 14. Teaching Summary

What students should learn from Task 1:

- **Discovery/EDA:** inspect data before editing.
- **Data structuring:** encode rules with dictionaries and graph structures.
- **Data quality:** distinguish accuracy errors, coverage anomalies, and outliers.
- **Cleansing:** prefer deterministic business rules when the assignment defines them.
- **Integration/enrichment:** use `branches.csv`, `nodes.csv`, and `edges.csv` to validate orders.
- **Validation:** final CSVs must be structurally correct and logically consistent.

A useful explanation sentence for tutoring:

> We are not simply changing suspicious values; we are using domain rules and reference data to produce a reproducible, validated wrangling pipeline.